<a href="https://colab.research.google.com/github/mAliAytekin/ai-research-notes/blob/main/clonalg_for_diophantine_equations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# --- 0. BAZI ÖNEMLİ IMPORTLAR ---

In [132]:
import random

# --- 1. PROBLEM TANIMI ---

> Blok alıntı ekle



In [133]:

# Denklem: a + 2b + 3c + 4d = 30
COEFFS = [1, 2, 3, 4] # a, b, c, d katsayıları
TARGET_RESULT = 30

BIT_LENGTH = 5  # Her değişken için 5 bit (0-31 arası)
GENE_LENGTH = BIT_LENGTH * len(COEFFS) # Toplam 20 bit


# --- 2. YARDIMCI FONKSİYONLAR ---

In [134]:


def binary_to_int(bit_list):
    bit_string = "".join(str(b) for b in bit_list)
    return int(bit_string, 2)

def antigen(cell):
    """Denklemin hatasını ve değişken değerlerini hesaplar."""
    values = []
    # Hücreyi dinamik olarak parçalara böl ve tam sayıya çevir
    for i in range(len(COEFFS)):
        start = i * BIT_LENGTH
        end = start + BIT_LENGTH
        values.append(binary_to_int(cell[start:end]))

    # Denklem sonucunu hesapla: (1*a + 2*b + 3*c + 4*d)
    current_result = sum(c * v for c, v in zip(COEFFS, values))

    error = abs(current_result - TARGET_RESULT)
    # Hata 0 ise affinity 1.0 olur.
    return 1.0 / (error + 1), values



# --- 3. TEMEL CLONALG ADIMLARI (SADELEŞTİRİLMİŞ) ---

In [135]:
pop_size = 50
num_generations = 500
population = [[random.randint(0, 1) for _ in range(GENE_LENGTH)] for _ in range(pop_size)]

solution_found = False
best_vals = []

print("Denklem çözülüyor...")

for gen in range(num_generations):
    # Uygunlukları hesapla
    pop_results = []
    for cell in population:
        fit, vals = antigen(cell)
        pop_results.append((cell, fit, vals))

        if fit == 1.0: # Tam çözüm bulundu
            best_cell, _, best_vals = (cell, fit, vals)
            solution_found = True
            break

    if solution_found: break

    # En iyiyi seç (Elitizm)
    pop_results.sort(key=lambda x: x[1], reverse=True)
    best_cell = pop_results[0][0]

    # Yeni popülasyon: En iyi hücre + onun mutasyonlu klonları
    new_population = [best_cell]
    for _ in range(pop_size - 1):
        clone = list(best_cell)
        # Mutasyon: Rastgele 1 veya 2 biti değiştir (Arama alanını genişletmek için)
        for _ in range(random.randint(1, 2)):
            idx = random.randint(0, GENE_LENGTH - 1)
            clone[idx] = 1 - clone[idx] # 0 ise 1, 1 ise 0 yapar
        new_population.append(clone)

    population = new_population

Denklem çözülüyor...


# --- 4. SONUÇLAR ---

In [136]:
if solution_found:
    print(f"\n{gen+1}. nesilde çözüm bulundu!")
    a, b, c, d = best_vals
    print(f"Sonuç: a={a}, b={b}, c={c}, d={d}")
    print(f"Kontrol: 1({a}) + 2({b}) + 3({c}) + 4({d}) = {1*a + 2*b + 3*c + 4*d}")
else:
    print("\nÇözüm bulunamadı, nesil sayısını veya popülasyonu artırmayı deneyin.")


3. nesilde çözüm bulundu!
Sonuç: a=8, b=3, c=4, d=1
Kontrol: 1(8) + 2(3) + 3(4) + 4(1) = 30
